In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pprint

import hydra
from omegaconf import OmegaConf
import polars as pl
import torch
import lightning.pytorch

from imagen_pytorch import ElucidatedImagen, ImagenTrainer, Unet3D

import conf.dataset
from conf import conf
from g_led import main_upsampler, datasets, utils

In [3]:
engine = conf.get_engine()
conf.orm.create_all(engine)
db = conf.sa.orm.Session(engine, expire_on_commit=False)
db.begin()

In [4]:
runs = pl.DataFrame(
    orient='row',
    schema=[
        'alt_id', 'label',
    ],
    data=[
        ['7dc5050a', 'model'],
    ]
)
runs

alt_id,label
str,str
"""7dc5050a""","""model"""


In [5]:
cfgs = db.execute(conf.sa.select(conf.Conf).where(conf.Conf.alt_id.in_(runs.get_column('alt_id'))))
cfgs = {cfg.alt_id: {'cfg': cfg} for (cfg,) in cfgs}
for alt_id in runs.get_column('alt_id'):
    cfg = cfgs[alt_id]['cfg']
    downsampler = datasets.Downsampler(cfg.dataset)
    upsampler = datasets.Upsampler(cfg.dataset)
    unet1 = Unet3D(
        dim=cfg.dataset.coarse_dimensions()[0],  # diff_args.unet_dim,
        cond_images_channels=cfg.dataset.solution_dimension,
        memory_efficient=True,
        dim_mults=(1, 2, 4, 8),  # mid: mid channel
    )
    width = cfg.dataset.dimensions()[0]
    imagen = ElucidatedImagen(
        unets=(unet1),
        image_sizes=width,
        image_width=width,
        channels=cfg.dataset.solution_dimension,   # Han Gao add the input to this args explicity
        random_crop_sizes=None,
        num_sample_steps=20,  # diff_args.num_sample_steps, # original is 10
        cond_drop_prob=0.1,
        sigma_min=0.002,
        sigma_max=80,      # max noise level, double the max noise level for upsampler (80, 160)
        sigma_data=0.5,      # standard deviation of data distribution
        rho=7,               # controls the sampling schedule
        P_mean=-1.2,         # mean of log-normal distribution from which noise is drawn for training
        P_std=1.2,           # standard deviation of log-normal distribution from which noise is drawn for training
        S_churn=80,          # parameters for stochastic sampling - depends on dataset, Table 5 in apper
        S_tmin=0.05,
        S_tmax=50,
        S_noise=1.003,
        condition_on_text=False,
        auto_normalize_img=False  # Han Gao make it false
    )
    imagen_trainer = ImagenTrainer(imagen, device=torch.device('cpu'))
    # imagen_trainer.load(path=cfg.run_dir/'last.ckpt')
    train_upsampler = main_upsampler.TrainUpsampler.load_from_checkpoint(
        cfg.run_dir/'last.ckpt',
        strict=False,
        cfg=cfg,
        downsampler=downsampler,
        upsampler=upsampler,
        imagen_trainer=imagen_trainer,
    )
    cfgs[alt_id]['train_upsampler'] = train_upsampler

The base dimension of your u-net should ideally be no smaller than 128, as recommended by a professional DDPM trainer https://nonint.com/2022/05/04/friends-dont-let-friends-train-small-diffusion-models/


/home/ttransue/GitHub/imagen-pytorch/imagen_pytorch/trainer.py:354: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled = grad_scaler_enabled)
/home/ttransue/GitHub/G-LED/.venv/lib/python3.10/site-packages/lightning/pytorch/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['imagen_trainer.unet_being_trained.null_text_embed', 'imagen_trainer.unet_being_trained.null_text_hidden', 'imagen_trainer.unet_being_trained.init_conv.convs.0.weight', 'imagen_trainer.unet_being_trained.init_conv.convs.0.bias', 'imagen_trainer.unet_being_trained.init_conv.convs.1.weight', 'imagen_trainer.unet_being_trained.init_conv.convs.1.bias', 'imagen_trainer.unet_being_trained.init_conv.convs.2.weight', 'imagen_trainer.unet_being_trained.init_conv.convs.2.bias', 'imagen_trainer.unet_being_trained.to_time_hiddens.0.weights', 'imagen_trainer.unet_being_trained.to_time_

In [6]:
with hydra.initialize(version_base=utils.HYDRA_INIT['version_base'], config_path='../conf'):
    cfg_dataset = hydra.compose(utils.HYDRA_INIT['config_name'], overrides=[
        'model=ImagenBackwardFacingStep2D',
        
        'dataset=BackwardFacingStep2D',
        'dataset.time_step_window_size_train=10',
        'dataset.time_step_window_size_val=10',
    ]).dataset
    cfg_dataset = conf.orm.instantiate_and_insert_config(db, OmegaConf.to_container(cfg_dataset, resolve=True))
    db.commit()
    pprint.pp(cfg_dataset)

BackwardFacingStep2D(_data_dir='/home/ttransue/out/g_led/data',
                     rng_seed=2376999025,
                     _processed_filename='x74xzrhr',
                     trajectory_count_train=1,
                     trajectory_count_val=1,
                     trajectory_count_test=1,
                     trajectories_are_shared_across_splits=True,
                     trajectory_time_step_size_micro=0.0002,
                     trajectory_time_step_count_micro=2498750,
                     trajectory_time_step_count_drop_first_micro=0,
                     trajectory_time_step_subsample_interval_macro=250,
                     macro_time_step_count_train=80,
                     macro_time_step_count_val=50,
                     macro_time_step_count_test=14,
                     time_step_window_size_train=10,
                     time_step_window_size_val=10,
                     time_step_window_size_test=10,
                     batch_size_train=16,
                    

In [7]:
lightning.pytorch.seed_everything(cfg.rng_seed)
with lightning.pytorch.utilities.seed.isolate_rng():
    dataset = datasets.get_dataset(cfg_dataset)
    dataset.prepare_data()
    dataset.setup('fit')

Seed set to 2376999025


In [8]:
dataset.train.shape

torch.Size([71, 10, 2, 512, 512])

In [9]:
for alt_id, info in cfgs.items():
    train_upsampler = info['train_upsampler']
    train_upsampler.upsample(train_upsampler.downsampler(dataset.train[:1]))

0it [00:00, ?it/s]

sampling time step:   0%|          | 0/20 [00:00<?, ?it/s]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f9876af0c40>>
Traceback (most recent call last):
  File "/home/ttransue/.cache/uv/archive-v0/FZpmNC0hAgQ-4cOFzLf3c/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 

KeyboardInterrupt

